In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l0.2 Tokens

Before a model sees text it sees **numbers**. PocketLM uses the simplest
possible scheme: one integer per character.

In [ ]:
from pocketlm import CharTokenizer

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
print("vocab size:", tok.vocab_size)
ids = tok.encode("To be")
print("encode('To be') ->", ids)
print("decode back   ->", repr(tok.decode(ids)))

## Three pathologies of character tokenization

In [ ]:
# 1) Out-of-vocab characters map to <unk>, so the round trip is lossy.
oov = "\u2603"  # a snowman, not in Shakespeare
print("OOV round trip:", repr(tok.decode(tok.encode(oov))))
# 2) An emoji is its own codepoint here, also out of vocab.
print("emoji ids:", tok.encode("\U0001f600"))
# 3) Char level means long sequences: one token per character.
print("'language' needs", len(tok.encode("language")), "tokens")

This is why real models use subword tokenization (BPE): fewer tokens and
no out-of-vocab holes. We will build up to that. For in-vocab text the
char round trip is exact:

In [ ]:
assert tok.decode(tok.encode("To be")) == "To be"